JSON PARSING AND PROCESSING

In [4]:
import json
import os

temporary json data

In [9]:
json_data = {
    "company_name": "TechNova Solutions",
    "employees": [
        {
            "id": "EMP001",
            "name": "Aarav Sharma",
            "role": "Software Engineer",
            "skills": ["Python", "FastAPI", "PostgreSQL", "Docker"],
            "projects": [
                {
                    "project_id": "PRJ101",
                    "project_name": "NotesPortal",
                    "status": "In Progress",
                    "duration_months": 8,
                    "tech_stack": ["Next.js", "Flask", "Supabase"],
                    "team_size": 5
                },
                {
                    "project_id": "PRJ102",
                    "project_name": "ChatWithPDF",
                    "status": "Completed",
                    "duration_months": 4,
                    "tech_stack": ["Python", "LangChain", "FAISS"],
                    "team_size": 3
                }
            ]
        },
        {
            "id": "EMP002",
            "name": "Priya Patel",
            "role": "Frontend Developer",
            "skills": ["React", "Next.js", "TypeScript", "Tailwind CSS"],
            "projects": [
                {
                    "project_id": "PRJ101",
                    "project_name": "NotesPortal",
                    "status": "In Progress",
                    "duration_months": 8,
                    "tech_stack": ["Next.js", "Flask", "Supabase"],
                    "team_size": 5
                }
            ]
        },
        {
            "id": "EMP003",
            "name": "Rahul Mehta",
            "role": "Backend Developer",
            "skills": ["Node.js", "Express", "MongoDB", "Redis"],
            "projects": [
                {
                    "project_id": "PRJ103",
                    "project_name": "Messenger App",
                    "status": "In Progress",
                    "duration_months": 6,
                    "tech_stack": ["React Native", "Express", "MongoDB"],
                    "team_size": 4
                }
            ]
        },
        {
            "id": "EMP004",
            "name": "Sneha Gupta",
            "role": "Data Scientist",
            "skills": ["Python", "Pandas", "Scikit-learn", "TensorFlow"],
            "projects": [
                {
                    "project_id": "PRJ104",
                    "project_name": "Survey Analytics",
                    "status": "Completed",
                    "duration_months": 5,
                    "tech_stack": ["Python", "Pandas", "Power BI"],
                    "team_size": 2
                },
                {
                    "project_id": "PRJ105",
                    "project_name": "Recommendation Engine",
                    "status": "Planning",
                    "duration_months": 7,
                    "tech_stack": ["Python", "TensorFlow", "Redis"],
                    "team_size": 6
                }
            ]
        }
    ]
}

creating json file

In [10]:
with open('data/json_files/company_data.json','w') as f:
    json.dump(json_data,f,indent=2)

In [7]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp":"2024-01-01","event":"user_login","user_id":123},
    {"timestamp":"2024-01-01","event":"page_view","user_id":123,"page":"/home"},
    {"timestamp":"2024-01-01","event":"purchase","user_id":123,"amount": 99.99}
]

with open('data/json_files/events.jsonl','w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item)+'\n')

JSON PROCESSING TECHNIQUES

In [8]:
from langchain_community.document_loaders import JSONLoader

Method 1: JSONLoader with jq_schema

In [11]:
json_loader = JSONLoader(
    file_path = 'data/json_files/company_data.json',
    jq_schema = '.employees[]', # jq query to extract each employee
    text_content = False 
)

employee_docs = json_loader.load()
print(f"No of employee docs: {len(employee_docs)}")
print(f"First employee : {employee_docs[0].page_content}")

No of employee docs: 4
First employee : {"id": "EMP001", "name": "Aarav Sharma", "role": "Software Engineer", "skills": ["Python", "FastAPI", "PostgreSQL", "Docker"], "projects": [{"project_id": "PRJ101", "project_name": "NotesPortal", "status": "In Progress", "duration_months": 8, "tech_stack": ["Next.js", "Flask", "Supabase"], "team_size": 5}, {"project_id": "PRJ102", "project_name": "ChatWithPDF", "status": "Completed", "duration_months": 4, "tech_stack": ["Python", "LangChain", "FAISS"], "team_size": 3}]}


Method 2: Custom JSON processing for complex structures

In [14]:
from typing import List
from langchain_core.documents import Document

def process_json(filepath: str) -> List[Document]:
    with open(filepath,'r') as f:
        data = json.load(f)
    
    documents = []
    
    for emp in data.get('employees',[]):
        content = f"""Employee Profile:
        Name: {emp['name']}
        Role: {emp['role']}
        Skills: {', '.join(emp['skills'])}
        Projects:"""
        for proj in emp.get('projects',[]):
            content += f"\n- {proj['project_name']} (Status: {proj['status']})"

        doc = Document(
            page_content = content,
            metadata = {
                'source': filepath,
                'data_type': 'emp_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )
        documents.append(doc)
    return documents

In [15]:
process_json('data/json_files/company_data.json')

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'emp_profile', 'employee_id': 'EMP001', 'employee_name': 'Aarav Sharma', 'role': 'Software Engineer'}, page_content='Employee Profile:\n        Name: Aarav Sharma\n        Role: Software Engineer\n        Skills: Python, FastAPI, PostgreSQL, Docker\n        Projects:\n- NotesPortal (Status: In Progress)\n- ChatWithPDF (Status: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'emp_profile', 'employee_id': 'EMP002', 'employee_name': 'Priya Patel', 'role': 'Frontend Developer'}, page_content='Employee Profile:\n        Name: Priya Patel\n        Role: Frontend Developer\n        Skills: React, Next.js, TypeScript, Tailwind CSS\n        Projects:\n- NotesPortal (Status: In Progress)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'emp_profile', 'employee_id': 'EMP003', 'employee_name': 'Rahul Mehta', 'role': 'Backend Developer'}, pa